# 05. Walk-forward Evaluation Protocol

## 목적

이 노트북에서는 **모델 성능을 올리지 않습니다.**

앞으로 비교할 MLP, XGBoost, CatBoost, Feature Set, Embedding이 모두 같은 기준으로 평가되도록 **시간축 검증 프로토콜**을 먼저 고정합니다.

핵심 원칙:

1. **Test(2024-25 → 2025-26)는 잠근다.** 이 노트북에서는 Test CSV를 읽지 않는다.
2. 미래 시즌이 과거 학습에 섞이지 않도록 **Expanding-window Walk-forward**를 사용한다.
3. 딥러닝의 Early Stopping에 Outer Validation을 직접 사용하지 않는다.
4. 각 Outer Fold 내부에서 **가장 최근 학습 시즌을 Inner Validation**으로 사용해 best epoch만 결정한다.
5. best epoch가 정해지면 Outer Train 전체로 다시 학습한 뒤 Outer Validation을 평가한다.
6. 전처리기(StandardScaler / OneHotEncoder)는 **각 Fold의 학습 데이터에만 fit**한다.

> 이 구조는 이후 노트북들의 공통 평가 규칙으로 재사용합니다.

## 0. 이번 노트북에서 사용할 모델

평가 프로토콜이 정상 작동하는지 확인하기 위해 **Exp9Cb 피처 + 단순 MLP**를 사용합니다.

아직 Architecture, Dropout, Learning Rate, Weight Decay 등을 튜닝하지 않습니다.

또한 sanity check용으로 `현재 시즌 goals를 그대로 다음 시즌 예측값으로 사용하는 Naive baseline`을 함께 기록합니다.

In [2]:
from pathlib import Path
import copy
import random
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings("ignore")

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def reset_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


reset_seed()
print("Device:", DEVICE)

Device: cpu


## 1. 파일 경로 설정

가능하면 기존 프로젝트 구조를 자동 탐색합니다.

자동 탐색이 실패하면 아래 `MANUAL_*_PATH`에 직접 경로를 지정하세요.

필요 파일:

- `long_basic/train.csv`
- `long_basic/validation.csv`
- `team_season_stats_2000_2024.csv`
- `team_name_alias_map.csv`

**Test CSV는 의도적으로 로드하지 않습니다.**

In [3]:
# 자동 탐색이 실패할 때만 직접 지정
MANUAL_TRAIN_PATH = None
MANUAL_VALIDATION_PATH = None
MANUAL_TEAM_STATS_PATH = None
MANUAL_ALIAS_PATH = None

SEARCH_ROOTS = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
]

# 중복 제거 + 실제 존재하는 경로만
SEARCH_ROOTS = list(dict.fromkeys(p.resolve() for p in SEARCH_ROOTS if p.exists()))

LONG_BASIC_REQUIRED = {
    "player", "team", "league", "season", "target_season",
    "position_group", "age", "matched_next", "starts", "minutes",
    "goals", "assists", "non_penalty_goals", "penalty_goals",
    "penalty_attempts", "goals_per90", "assists_per90",
    "goal_contrib_per90", "next_goals"
}

ADVANCED_FORBIDDEN = {"xg", "xag", "shots", "progressive_carries"}


def read_header(path):
    try:
        return set(pd.read_csv(path, nrows=0).columns)
    except Exception:
        return set()


def find_csv_by_schema(kind):
    candidates = []

    for root in SEARCH_ROOTS:
        try:
            for path in root.rglob("*.csv"):
                # 같은 파일을 너무 넓게 반복 탐색하는 것을 줄임
                if any(part.startswith(".") for part in path.parts):
                    continue

                cols = read_header(path)

                if kind in {"train", "validation"}:
                    if not LONG_BASIC_REQUIRED.issubset(cols):
                        continue
                    if ADVANCED_FORBIDDEN.intersection(cols):
                        continue

                    # 이름 + 시즌을 함께 확인
                    name = path.name.lower()
                    sample = pd.read_csv(path, usecols=["season"], nrows=50)
                    seasons = set(sample["season"].astype(str))

                    if kind == "train":
                        score = int("train" in name) + int("2000-2001" in seasons)
                    else:
                        score = int("validation" in name or "val" in name) + int("2023-2024" in seasons)

                    candidates.append((score, path))

                elif kind == "team_stats":
                    needed = {
                        "league", "season", "team_name", "team_rank_pct",
                        "team_points_per_game", "team_goal_diff_per_game"
                    }
                    if needed.issubset(cols):
                        candidates.append((2 if "team_season_stats" in path.name.lower() else 1, path))

                elif kind == "alias":
                    needed = {"player_data_team", "standings_team"}
                    if needed.issubset(cols):
                        candidates.append((2 if "alias" in path.name.lower() else 1, path))
        except (PermissionError, OSError):
            continue

    if not candidates:
        return None

    candidates.sort(key=lambda x: (x[0], -len(str(x[1]))), reverse=True)
    return candidates[0][1]


def resolve_path(manual_path, kind):
    if manual_path is not None:
        path = Path(manual_path)
        if not path.exists():
            raise FileNotFoundError(f"지정한 경로가 없습니다: {path}")
        return path

    path = find_csv_by_schema(kind)
    if path is None:
        raise FileNotFoundError(
            f"{kind} 파일을 자동으로 찾지 못했습니다. "
            f"위 MANUAL_*_PATH에 경로를 직접 지정하세요."
        )
    return path


TRAIN_PATH = resolve_path(MANUAL_TRAIN_PATH, "train")
VALIDATION_PATH = resolve_path(MANUAL_VALIDATION_PATH, "validation")
TEAM_STATS_PATH = resolve_path(MANUAL_TEAM_STATS_PATH, "team_stats")
ALIAS_PATH = resolve_path(MANUAL_ALIAS_PATH, "alias")

print("TRAIN      :", TRAIN_PATH)
print("VALIDATION :", VALIDATION_PATH)
print("TEAM STATS :", TEAM_STATS_PATH)
print("ALIAS       :", ALIAS_PATH)

TRAIN      : D:\dev\03_PersonalProjects\next_season_goal_prediction\data\long_basic\train.csv
VALIDATION : D:\dev\03_PersonalProjects\next_season_goal_prediction\data\long_basic\validation.csv
TEAM STATS : D:\dev\03_PersonalProjects\next_season_goal_prediction\data\team_season_stats_2000_2024.csv
ALIAS       : D:\dev\03_PersonalProjects\next_season_goal_prediction\data\team_name_alias_map.csv


### 체크 포인트

아래 출력에서 반드시 확인합니다.

- Train은 2000-01부터 시작하는 `long_basic`
- Validation은 `2023-2024 → 2024-2025`
- xG 등의 advanced 컬럼이 없어야 함

In [4]:
train_raw = pd.read_csv(TRAIN_PATH)
validation_raw = pd.read_csv(VALIDATION_PATH)
team_stats_raw = pd.read_csv(TEAM_STATS_PATH)
team_alias_raw = pd.read_csv(ALIAS_PATH)

# 저장 과정에서 생긴 인덱스 컬럼 제거
for df in [train_raw, validation_raw, team_stats_raw, team_alias_raw]:
    drop_cols = [c for c in df.columns if c.lower().startswith("unnamed") or c == "index"]
    if drop_cols:
        df.drop(columns=drop_cols, inplace=True)

print("Train shape      :", train_raw.shape)
print("Validation shape :", validation_raw.shape)
print("Train seasons    :", train_raw["season"].min(), "~", train_raw["season"].max())
print("Validation season:", validation_raw["season"].unique())
print("Advanced columns present:", sorted(ADVANCED_FORBIDDEN.intersection(train_raw.columns)))

Train shape      : (22430, 22)
Validation shape : (923, 22)
Train seasons    : 2000-2001 ~ 2022-2023
Validation season: <StringArray>
['2023-2024']
Length: 1, dtype: str
Advanced columns present: []


## 2. Test Lock 선언

현재 프로젝트의 최종 Test는:

- 입력 시즌: `2024-2025`
- 타깃 시즌: `2025-2026`

이 노트북에서는 Test 파일을 읽지 않습니다.

개발 데이터는 `Train + 기존 Validation(2023-24)`까지만 사용합니다.

In [5]:
LOCKED_TEST_INPUT_SEASON = "2024-2025"
LOCKED_TEST_TARGET_SEASON = "2025-2026"

assert LOCKED_TEST_INPUT_SEASON not in set(train_raw["season"].astype(str))
assert LOCKED_TEST_INPUT_SEASON not in set(validation_raw["season"].astype(str))

print("Locked Test:", LOCKED_TEST_INPUT_SEASON, "→", LOCKED_TEST_TARGET_SEASON)
print("Test CSV is NOT loaded in this notebook.")

Locked Test: 2024-2025 → 2025-2026
Test CSV is NOT loaded in this notebook.


## 3. 개발용 전체 데이터 구성

Walk-forward에서는 과거 여러 시즌을 다시 Validation으로 사용해야 하므로 기존 Train과 Validation을 합칩니다.

중요:

- 아직 `matched_next=True`로 필터링하지 않습니다.
- 과거 득점 파생 피처를 만들 때는 **현재/과거에 실제 관측된 선수 기록 전체**가 필요합니다.
- 미래 정보인 `matched_next`는 모델 입력으로 사용하지 않습니다.

In [6]:
dev_raw = pd.concat(
    [train_raw, validation_raw],
    ignore_index=True
).copy()


def season_start_value(series):
    return series.astype(str).str[:4].astype(int)


dev_raw["season_start"] = season_start_value(dev_raw["season"])
dev_raw = dev_raw.sort_values(["player", "season_start"]).reset_index(drop=True)

print("Development rows:", len(dev_raw))
print("Season range:", dev_raw["season"].min(), "~", dev_raw["season"].max())
print("Target range:", dev_raw["target_season"].min(), "~", dev_raw["target_season"].max())

Development rows: 23353
Season range: 2000-2001 ~ 2023-2024
Target range: 2001-2002 ~ 2024-2025


## 4. Exp9Cb Historical Features 재생성

Exp9Cb에서 채택한 최근 scoring trend:

- `goals_3yr_mean`
- `goals_per90_3yr_mean`
- `goals_3yr_max`

반드시 `shift(1)` 후 rolling 하여 **현재 시즌 값을 자기 자신의 과거 피처에 포함시키지 않습니다.**

In [7]:
# player별 시간순 정렬 상태에서 과거 관측치만 사용
player_group = dev_raw.groupby("player", sort=False)


dev_raw["goals_3yr_mean"] = player_group["goals"].transform(
    lambda s: s.shift(1).rolling(window=3, min_periods=1).mean()
)

dev_raw["goals_per90_3yr_mean"] = player_group["goals_per90"].transform(
    lambda s: s.shift(1).rolling(window=3, min_periods=1).mean()
)

dev_raw["goals_3yr_max"] = player_group["goals"].transform(
    lambda s: s.shift(1).rolling(window=3, min_periods=1).max()
)

historical_cols = [
    "goals_3yr_mean",
    "goals_per90_3yr_mean",
    "goals_3yr_max"
]

dev_raw[historical_cols] = dev_raw[historical_cols].fillna(0.0)

print(dev_raw[historical_cols].describe().T[["count", "mean", "std", "min", "max"]])

                        count      mean       std  min        max
goals_3yr_mean        23353.0  3.241025  4.253875  0.0  42.333333
goals_per90_3yr_mean  23353.0  0.140868  0.172466  0.0   1.315336
goals_3yr_max         23353.0  4.363551  5.545552  0.0  50.000000


## 5. 팀 성적 데이터 Merge

Exp9Cb에서 최종적으로 채택한 팀 피처:

- `team_rank_pct`
- `team_points_per_game`
- `team_goal_diff_per_game`

팀명 alias를 적용한 뒤 `league + season + team_name`으로 연결합니다.

In [8]:
team_name_map = dict(zip(
    team_alias_raw["player_data_team"],
    team_alias_raw["standings_team"]
))

# 기존 실험에서 추가로 확인된 alias
team_name_map.update({
    "Gladbach": "M'gladbach",
    "Luton Town": "Luton",
})


dev = dev_raw.copy()
dev["team_name"] = dev["team"].replace(team_name_map)

team_stats = team_stats_raw.copy()
if "position" in team_stats.columns and "team_rank" not in team_stats.columns:
    team_stats = team_stats.rename(columns={"position": "team_rank"})

team_keep_cols = [
    "league", "season", "team_name",
    "team_rank_pct",
    "team_points_per_game",
    "team_goal_diff_per_game"
]

team_stats = team_stats[team_keep_cols].drop_duplicates(
    subset=["league", "season", "team_name"]
)

dev = dev.merge(
    team_stats,
    on=["league", "season", "team_name"],
    how="left",
    validate="many_to_one"
)

team_feature_cols = [
    "team_rank_pct",
    "team_points_per_game",
    "team_goal_diff_per_game"
]

match_ratio = dev[team_feature_cols].notna().all(axis=1).mean()
print(f"Team feature match ratio: {match_ratio:.4%}")

unmatched = dev.loc[
    dev[team_feature_cols].isna().any(axis=1),
    ["league", "season", "team", "team_name"]
].drop_duplicates()

if len(unmatched):
    display(unmatched.head(30))

assert match_ratio > 0.99, (
    "팀 성적 매칭률이 99% 이하입니다. alias 또는 파일을 확인하세요."
)

Team feature match ratio: 100.0000%


## 6. 조건부 회귀 대상 확정

현재 회귀 문제는:

> **다음 시즌에도 5대 리그 기록이 존재하는 선수(`matched_next=True`)의 다음 시즌 득점 수 예측**

따라서 Feature 생성이 끝난 뒤 `matched_next=True`만 남깁니다.

`matched_next` 자체는 미래 정보이므로 입력 피처에 절대 포함하지 않습니다.

In [9]:
model_df = dev[dev["matched_next"] == True].copy()
model_df = model_df.sort_values(["season_start", "player"]).reset_index(drop=True)

print("Conditional regression rows:", len(model_df))
print("Seasons:", model_df["season"].min(), "~", model_df["season"].max())
print("Target mean:", model_df["next_goals"].mean())
print("Zero ratio:", (model_df["next_goals"] == 0).mean())

Conditional regression rows: 19403
Seasons: 2000-2001 ~ 2023-2024
Target mean: 3.5985156934494666
Zero ratio: 0.26897902386228933


## 7. Exp9Cb Feature Set 고정

이 노트북에서는 Feature Set을 바꾸지 않습니다.

### Numeric

현재 시즌 기본 기록 + per90 + 최근 scoring trend + 팀 종합 성적

### Categorical

- league
- position_group

예상 전처리 후 차원은 약 24입니다.

In [10]:
BASE_NUMERIC = [
    "age",
    "starts",
    "minutes",
    "goals",
    "assists",
    "non_penalty_goals",
    "penalty_goals",
    "penalty_attempts",
    "goals_per90",
    "assists_per90",
    "goal_contrib_per90",
]

HISTORICAL_NUMERIC = [
    "goals_3yr_mean",
    "goals_per90_3yr_mean",
    "goals_3yr_max",
]

TEAM_NUMERIC = [
    "team_rank_pct",
    "team_points_per_game",
    "team_goal_diff_per_game",
]

FINAL_NUMERIC = BASE_NUMERIC + HISTORICAL_NUMERIC + TEAM_NUMERIC
FINAL_CATEGORICAL = ["league", "position_group"]
TARGET = "next_goals"

print("Numeric count:", len(FINAL_NUMERIC))
print("Categorical:", FINAL_CATEGORICAL)

Numeric count: 17
Categorical: ['league', 'position_group']


## 8. Outer Walk-forward Fold 설계

마지막 4개 개발 시즌을 순차적으로 Outer Validation으로 사용합니다.

예시:

- Train ≤ 2019-20 → Validate 2020-21 (target 2021-22)
- Train ≤ 2020-21 → Validate 2021-22
- Train ≤ 2021-22 → Validate 2022-23
- Train ≤ 2022-23 → Validate 2023-24

각 Fold에서 **Validation 시즌 이후의 데이터는 절대 사용하지 않습니다.**

In [11]:
OUTER_VAL_SEASONS = [
    "2020-2021",
    "2021-2022",
    "2022-2023",
    "2023-2024",
]

available_seasons = set(model_df["season"].astype(str))
missing_outer = [s for s in OUTER_VAL_SEASONS if s not in available_seasons]
assert not missing_outer, f"Outer validation season 누락: {missing_outer}"

fold_plan = []

for fold_id, val_season in enumerate(OUTER_VAL_SEASONS, start=1):
    val_start = int(val_season[:4])
    outer_train = model_df[model_df["season_start"] < val_start]
    outer_val = model_df[model_df["season"] == val_season]

    latest_train_start = outer_train["season_start"].max()
    inner_val_season = outer_train.loc[
        outer_train["season_start"] == latest_train_start,
        "season"
    ].iloc[0]

    fold_plan.append({
        "fold": fold_id,
        "outer_train_end": outer_train["season"].iloc[-1] if len(outer_train) else None,
        "inner_val_season": inner_val_season,
        "outer_val_season": val_season,
        "target_season": outer_val["target_season"].iloc[0],
        "outer_train_n": len(outer_train),
        "outer_val_n": len(outer_val),
    })

fold_plan_df = pd.DataFrame(fold_plan)
fold_plan_df

,fold,outer_train_end,inner_val_season,outer_val_season,target_season,outer_train_n,outer_val_n
0,1,2019-2020,2019-2020,2020-2021,2021-2022,16284,834
1,2,2020-2021,2020-2021,2021-2022,2022-2023,17118,778
2,3,2021-2022,2021-2022,2022-2023,2023-2024,17896,738
3,4,2022-2023,2022-2023,2023-2024,2024-2025,18634,769


### 누수 검증

각 Fold에서 다음 조건을 assert 합니다.

- `max(Outer Train season) < Outer Validation season`
- `Inner Validation`은 Outer Train 내부의 마지막 시즌
- 전처리 fit 대상은 Inner Train 또는 Outer Train뿐

In [12]:
for val_season in OUTER_VAL_SEASONS:
    val_start = int(val_season[:4])
    outer_train = model_df[model_df["season_start"] < val_start]
    outer_val = model_df[model_df["season_start"] == val_start]

    assert outer_train["season_start"].max() < outer_val["season_start"].min()

print("Temporal leakage assertions passed.")

Temporal leakage assertions passed.


## 9. Fold별 전처리 함수

OneHotEncoder의 범주는 미래 Validation에서 새로 등장할 수 있으므로 `handle_unknown='ignore'`를 사용합니다.

전처리기는 반드시 학습 데이터에만 fit합니다.

In [13]:
def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), FINAL_NUMERIC),
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                FINAL_CATEGORICAL,
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=True,
    )


def fit_transform_split(train_df, eval_df):
    preprocessor = make_preprocessor()

    X_train = preprocessor.fit_transform(
        train_df[FINAL_NUMERIC + FINAL_CATEGORICAL]
    )
    X_eval = preprocessor.transform(
        eval_df[FINAL_NUMERIC + FINAL_CATEGORICAL]
    )

    y_train = train_df[TARGET].to_numpy(dtype=np.float32)
    y_eval = eval_df[TARGET].to_numpy(dtype=np.float32)

    return X_train, X_eval, y_train, y_eval, preprocessor

## 10. 고정 Baseline MLP

여기서는 04에서 사용한 단순 구조를 그대로 사용합니다.

- 64 → 32 → 1
- ReLU
- Adam
- MSE

Early Stopping은 모델 튜닝이 아니라 **각 Fold에서 필요한 epoch 수를 정하기 위한 학습 절차**로만 사용합니다.

In [14]:
class BaselineMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x)

## 11. 학습 유틸리티

### Step A: Inner Validation으로 best epoch 결정

Outer Train 안에서 가장 최근 시즌 하나를 Inner Validation으로 둡니다.

### Step B: Outer Train 전체 재학습

best epoch가 정해지면 Inner Validation도 다시 학습 데이터에 포함하고, Outer Train 전체로 정확히 `best_epoch`만큼 재학습합니다.

그 후에만 Outer Validation을 평가합니다.

In [15]:
def make_loader(X, y, batch_size=64, shuffle=False):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)
    ds = TensorDataset(X_t, y_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def find_best_epoch(
    inner_train_df,
    inner_val_df,
    max_epochs=100,
    patience=10,
    lr=1e-3,
    batch_size=64,
    seed=SEED,
):
    reset_seed(seed)

    X_train, X_val, y_train, y_val, _ = fit_transform_split(
        inner_train_df, inner_val_df
    )

    train_loader = make_loader(X_train, y_train, batch_size, shuffle=True)
    val_loader = make_loader(X_val, y_val, batch_size, shuffle=False)

    model = BaselineMLP(X_train.shape[1]).to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val = float("inf")
    best_epoch = 1
    patience_count = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_loss_sum = 0.0

        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            optimizer.zero_grad()
            pred = model(X_b)
            loss = criterion(pred, y_b)
            loss.backward()
            optimizer.step()
            train_loss_sum += loss.item() * len(X_b)

        train_loss = train_loss_sum / len(train_loader.dataset)

        model.eval()
        val_loss_sum = 0.0
        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
                pred = model(X_b)
                loss = criterion(pred, y_b)
                val_loss_sum += loss.item() * len(X_b)

        val_loss = val_loss_sum / len(val_loader.dataset)
        history.append((epoch, train_loss, val_loss))

        if val_loss < best_val - 1e-8:
            best_val = val_loss
            best_epoch = epoch
            patience_count = 0
        else:
            patience_count += 1

        if patience_count >= patience:
            break

    return best_epoch, best_val, pd.DataFrame(
        history, columns=["epoch", "train_mse", "inner_val_mse"]
    )


def train_fixed_epochs(
    outer_train_df,
    outer_val_df,
    epochs,
    lr=1e-3,
    batch_size=64,
    seed=SEED,
):
    reset_seed(seed)

    X_train, X_val, y_train, y_val, preprocessor = fit_transform_split(
        outer_train_df, outer_val_df
    )

    train_loader = make_loader(X_train, y_train, batch_size, shuffle=True)

    model = BaselineMLP(X_train.shape[1]).to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for _ in range(epochs):
        model.train()
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            optimizer.zero_grad()
            pred = model(X_b)
            loss = criterion(pred, y_b)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        pred = model(
            torch.tensor(X_val, dtype=torch.float32, device=DEVICE)
        ).cpu().numpy().ravel()

    return model, preprocessor, y_val, pred, X_train.shape[1]

## 12. 평가 함수

Primary metric은 우선 기존 실험과 동일하게 MAE / RMSE / R²를 모두 기록합니다.

어떤 지표를 모델 선택의 최우선 기준으로 사용할지는 이후 목적 정의에서 결정합니다.

In [16]:
def regression_metrics(y_true, y_pred):
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
        "r2": r2_score(y_true, y_pred),
        "bias": float(np.mean(y_pred - y_true)),
    }


def high_scorer_metrics(y_true, y_pred, threshold=10):
    mask = y_true >= threshold
    if mask.sum() == 0:
        return {
            "count": 0,
            "mae": np.nan,
            "bias": np.nan,
            "actual_mean": np.nan,
            "pred_mean": np.nan,
        }

    return {
        "count": int(mask.sum()),
        "mae": mean_absolute_error(y_true[mask], y_pred[mask]),
        "bias": float(np.mean(y_pred[mask] - y_true[mask])),
        "actual_mean": float(np.mean(y_true[mask])),
        "pred_mean": float(np.mean(y_pred[mask])),
    }

## 13. Walk-forward 실행

각 Outer Fold에서:

1. Outer Train / Outer Validation 분리
2. Outer Train의 마지막 시즌을 Inner Validation으로 분리
3. Inner Validation으로 best epoch 결정
4. Outer Train 전체로 best epoch만큼 재학습
5. Outer Validation 평가
6. Naive baseline도 같은 Outer Validation에서 평가

**주의:** 실행 시간이 조금 걸릴 수 있습니다.

In [17]:
fold_results = []
fold_predictions = []
inner_histories = {}

for fold_id, outer_val_season in enumerate(OUTER_VAL_SEASONS, start=1):
    outer_val_start = int(outer_val_season[:4])

    outer_train_df = model_df[
        model_df["season_start"] < outer_val_start
    ].copy()

    outer_val_df = model_df[
        model_df["season_start"] == outer_val_start
    ].copy()

    # Outer Train 안의 마지막 시즌 = Inner Validation
    inner_val_start = outer_train_df["season_start"].max()

    inner_train_df = outer_train_df[
        outer_train_df["season_start"] < inner_val_start
    ].copy()

    inner_val_df = outer_train_df[
        outer_train_df["season_start"] == inner_val_start
    ].copy()

    assert inner_train_df["season_start"].max() < inner_val_df["season_start"].min()
    assert outer_train_df["season_start"].max() < outer_val_df["season_start"].min()

    inner_val_season = inner_val_df["season"].iloc[0]

    best_epoch, best_inner_mse, history = find_best_epoch(
        inner_train_df,
        inner_val_df,
        max_epochs=100,
        patience=10,
        lr=1e-3,
        batch_size=64,
        seed=SEED + fold_id,
    )

    inner_histories[fold_id] = history

    model, preprocessor, y_true, y_pred, input_dim = train_fixed_epochs(
        outer_train_df,
        outer_val_df,
        epochs=best_epoch,
        lr=1e-3,
        batch_size=64,
        seed=SEED + fold_id,
    )

    mlp_m = regression_metrics(y_true, y_pred)
    high10 = high_scorer_metrics(y_true, y_pred, threshold=10)

    # Naive baseline: 현재 시즌 goals 그대로 다음 시즌 goals로 예측
    naive_pred = outer_val_df["goals"].to_numpy(dtype=float)
    naive_m = regression_metrics(y_true, naive_pred)

    fold_results.append({
        "fold": fold_id,
        "inner_val_season": inner_val_season,
        "outer_val_season": outer_val_season,
        "target_season": outer_val_df["target_season"].iloc[0],
        "train_n": len(outer_train_df),
        "val_n": len(outer_val_df),
        "input_dim": input_dim,
        "best_epoch": best_epoch,
        "best_inner_mse": best_inner_mse,
        "mlp_mae": mlp_m["mae"],
        "mlp_rmse": mlp_m["rmse"],
        "mlp_r2": mlp_m["r2"],
        "mlp_bias": mlp_m["bias"],
        "mlp_10plus_n": high10["count"],
        "mlp_10plus_mae": high10["mae"],
        "mlp_10plus_bias": high10["bias"],
        "naive_mae": naive_m["mae"],
        "naive_rmse": naive_m["rmse"],
        "naive_r2": naive_m["r2"],
    })

    pred_df = outer_val_df[
        ["player", "team", "league", "season", "target_season", "position_group", "goals", "next_goals"]
    ].copy()
    pred_df["fold"] = fold_id
    pred_df["mlp_pred"] = y_pred
    pred_df["naive_pred"] = naive_pred
    fold_predictions.append(pred_df)

    print(
        f"Fold {fold_id} | "
        f"Outer Val {outer_val_season} → {outer_val_df['target_season'].iloc[0]} | "
        f"Best Epoch {best_epoch} | "
        f"MLP MAE {mlp_m['mae']:.4f} | "
        f"Naive MAE {naive_m['mae']:.4f}"
    )

walk_forward_results = pd.DataFrame(fold_results)
walk_forward_predictions = pd.concat(fold_predictions, ignore_index=True)

Fold 1 | Outer Val 2020-2021 → 2021-2022 | Best Epoch 15 | MLP MAE 2.3835 | Naive MAE 2.6295
Fold 2 | Outer Val 2021-2022 → 2022-2023 | Best Epoch 2 | MLP MAE 2.5024 | Naive MAE 2.9229
Fold 3 | Outer Val 2022-2023 → 2023-2024 | Best Epoch 21 | MLP MAE 2.3604 | Naive MAE 2.7575
Fold 4 | Outer Val 2023-2024 → 2024-2025 | Best Epoch 10 | MLP MAE 2.4643 | Naive MAE 2.8114


## 14. 시즌별 결과 확인

여기서 중요한 것은 단순 평균뿐 아니라 **시즌별 변동성**입니다.

한 시즌에서만 잘 맞고 다른 시즌에서 크게 무너지면 운영 관점에서는 불안정한 모델일 수 있습니다.

In [18]:
walk_forward_results[
    [
        "fold",
        "outer_val_season",
        "target_season",
        "train_n",
        "val_n",
        "best_epoch",
        "mlp_mae",
        "mlp_rmse",
        "mlp_r2",
        "mlp_10plus_mae",
        "mlp_10plus_bias",
        "naive_mae",
    ]
]

,fold,outer_val_season,target_season,train_n,val_n,best_epoch,mlp_mae,mlp_rmse,mlp_r2,mlp_10plus_mae,mlp_10plus_bias,naive_mae
0,1,2020-2021,2021-2022,16284,834,15,2.383471,3.352308,0.484540,5.716736,-5.161948,2.629496
1,2,2021-2022,2022-2023,17118,778,2,2.502450,3.545215,0.404357,6.287442,-5.662632,2.922879
2,3,2022-2023,2023-2024,17896,738,21,2.360409,3.380157,0.474456,6.043118,-5.941598,2.757453
3,4,2023-2024,2024-2025,18634,769,10,2.464347,3.524136,0.450861,6.756019,-6.390883,2.811443


## 15. 평균 + 표준편차

앞으로 모델 비교에서는 최소한 다음을 같이 봅니다.

- Walk-forward 평균 성능
- Fold 간 표준편차
- 최악 Fold
- 고득점자 성능

단일 Validation 시즌 점수 하나만으로 모델을 선택하지 않습니다.

In [19]:
summary_rows = []

for model_name, prefix in [
    ("Baseline_MLP", "mlp"),
    ("Naive_current_goals", "naive"),
]:
    row = {"model": model_name}
    for metric in ["mae", "rmse", "r2"]:
        col = f"{prefix}_{metric}"
        row[f"{metric}_mean"] = walk_forward_results[col].mean()
        row[f"{metric}_std"] = walk_forward_results[col].std(ddof=1)
        row[f"{metric}_worst"] = (
            walk_forward_results[col].max()
            if metric in {"mae", "rmse"}
            else walk_forward_results[col].min()
        )
    summary_rows.append(row)

walk_forward_summary = pd.DataFrame(summary_rows)
walk_forward_summary

,model,mae_mean,mae_std,mae_worst,rmse_mean,rmse_std,rmse_worst,r2_mean,r2_std,r2_worst
0,Baseline_MLP,2.427669,0.066870,2.502450,3.450454,0.09829,3.545215,0.453554,0.035705,0.404357
1,Naive_current_goals,2.780318,0.121878,2.922879,4.033013,0.18078,4.301910,0.251752,0.086593,0.122952


## 16. Fold별 학습 곡선 확인

Early Stopping의 best epoch가 시즌마다 얼마나 달라지는지 확인합니다.

이 값이 크게 흔들린다면 이후 MLP 튜닝 시 학습 안정성을 별도로 고려해야 합니다.

In [20]:
for fold_id, history in inner_histories.items():
    best_idx = history["inner_val_mse"].idxmin()
    best_row = history.loc[best_idx]
    print(
        f"Fold {fold_id}: "
        f"best epoch={int(best_row['epoch'])}, "
        f"inner val MSE={best_row['inner_val_mse']:.4f}, "
        f"epochs executed={len(history)}"
    )

Fold 1: best epoch=15, inner val MSE=11.5865, epochs executed=25
Fold 2: best epoch=2, inner val MSE=11.1717, epochs executed=12
Fold 3: best epoch=21, inner val MSE=12.1278, epochs executed=31
Fold 4: best epoch=10, inner val MSE=11.3240, epochs executed=20


## 17. 고득점자 Walk-forward 진단

기존 분석에서 10+ 득점 선수를 크게 과소예측하는 현상이 확인되었습니다.

Walk-forward에서도 같은 문제가 여러 시즌에 반복되는지 확인합니다.

In [21]:
high_score_fold = walk_forward_results[
    [
        "fold",
        "outer_val_season",
        "target_season",
        "mlp_10plus_n",
        "mlp_10plus_mae",
        "mlp_10plus_bias",
    ]
].copy()

high_score_fold

,fold,outer_val_season,target_season,mlp_10plus_n,mlp_10plus_mae,mlp_10plus_bias
0,1,2020-2021,2021-2022,92,5.716736,-5.161948
1,2,2021-2022,2022-2023,76,6.287442,-5.662632
2,3,2022-2023,2023-2024,78,6.043118,-5.941598
3,4,2023-2024,2024-2025,87,6.756019,-6.390883


## 18. 전체 OOF 형태 예측 저장

각 Fold의 Outer Validation 예측을 이어 붙이면 개발 기간에 대한 **시간축 OOF-like prediction**을 얻습니다.

이 값은 이후 모델 비교 및 Slice Analysis에 재사용할 수 있습니다.

> 일반적인 랜덤 K-Fold OOF와는 다르고, 시간 순서를 지킨 Walk-forward prediction입니다.

In [22]:
print("Prediction rows:", len(walk_forward_predictions))
print("Validation seasons:", sorted(walk_forward_predictions["season"].unique()))

walk_forward_predictions.head()

Prediction rows: 3119
Validation seasons: ['2020-2021', '2021-2022', '2022-2023', '2023-2024']


,player,team,league,season,target_season,position_group,goals,next_goals,fold,mlp_pred,naive_pred
0,Aaron Ramsey,Juventus,Serie A,2020-2021,2021-2022,MF,2.0,0.0,1,2.684675,2.0
1,Abdoulaye Doucouré,Everton,Premier League,2020-2021,2021-2022,MF,2.0,2.0,1,2.791583,2.0
2,Abdoulaye Touré,Nantes,Ligue 1,2020-2021,2021-2022,MF,2.0,0.0,1,1.853818,2.0
3,Adam Lallana,Brighton,Premier League,2020-2021,2021-2022,FW,1.0,0.0,1,1.906186,1.0
4,Adam Ounas,Crotone,Serie A,2020-2021,2021-2022,FW,4.0,0.0,1,3.758674,4.0


## 19. Evaluation Protocol 저장용 설정

이 셀은 이후 06~11에서 동일한 평가 규칙을 재사용하기 위한 메타정보입니다.

In [23]:
EVALUATION_PROTOCOL = {
    "problem": "conditional_next_big5_goals_regression",
    "test_input_season": LOCKED_TEST_INPUT_SEASON,
    "test_target_season": LOCKED_TEST_TARGET_SEASON,
    "outer_validation_seasons": OUTER_VAL_SEASONS,
    "outer_scheme": "expanding_window",
    "inner_validation": "latest season inside each outer train",
    "preprocessing_fit_scope": "training partition only",
    "primary_metrics": ["MAE", "RMSE", "R2"],
    "diagnostic_metrics": ["bias", "10plus_MAE", "10plus_bias"],
}

EVALUATION_PROTOCOL

{'problem': 'conditional_next_big5_goals_regression',
 'test_input_season': '2024-2025',
 'test_target_season': '2025-2026',
 'outer_validation_seasons': ['2020-2021',
  '2021-2022',
  '2022-2023',
  '2023-2024'],
 'outer_scheme': 'expanding_window',
 'inner_validation': 'latest season inside each outer train',
 'preprocessing_fit_scope': 'training partition only',
 'primary_metrics': ['MAE', 'RMSE', 'R2'],
 'diagnostic_metrics': ['bias', '10plus_MAE', '10plus_bias']}

# 20. 결과 해석 — 직접 작성

아래는 **의도적으로 비워둡니다.**

## 확인할 질문

1. 단일 2023-24 Validation에서 보았던 성능과 Walk-forward 평균은 얼마나 다른가?
2. Fold별 MAE/RMSE/R² 변동폭은 큰가?
3. 어떤 시즌에서 가장 성능이 나빴는가? 그 시즌의 선수 분포에 특징이 있는가?
4. Baseline MLP가 `현재 goals 그대로 예측`하는 Naive baseline을 안정적으로 이기는가?
5. 고득점자 10+ bias는 모든 시즌에서 음수인가?
6. best epoch가 Fold마다 크게 달라지는가?
7. 이 평가 프로토콜을 06의 ML/DL 모델 비교 기준으로 채택해도 되는가?

## 내 결론

1. MLP는 Naive baseline보다 일관되게 좋은가?
→ 현재 결과상 Yes

2. 특정 시즌 하나에만 잘 맞는 모델인가?
→ 현재 결과상 No에 가까움

3. Early Stopping이 필요한가?
→ 매우 필요해 보임

4. 모델의 반복적인 실패 패턴은 무엇인가?
→ 고득점자 과소예측

5. 다음 모델 비교에서도 동일한 평가 프로토콜을 써야 하는가?
→ Yes
- 추가 확인이 필요한 점:

## 다음 단계

이 노트북의 평가 프로토콜이 정상 동작하는 것을 확인한 후:

### 06. ML / DL Baseline Comparison

동일한 Outer Walk-forward 기준에서 다음 후보를 비교합니다.

- Naive baseline
- Linear / ElasticNet
- Random Forest
- XGBoost
- CatBoost
- 현재 MLP

**06에서도 Test는 열지 않습니다.**